In [1]:
import os  # Used for file management (removing files, path operations)
import pandas as pd  # Used for handling tabular data (reading/writing CSV files)
import subprocess  # Used to run PHANOTATE as an external process
from tqdm import tqdm  # Used to display a progress bar for tracking processing
from os import listdir  # Used to list files in a directory
from Bio import SeqIO  # Used to read and parse FASTA files
from Bio.Seq import Seq  # Used for reverse complementing sequences

In [2]:
dir_path = '/home/dylan33smith/projects/Yuzhen/PB_interactions/'
phage_file = 'data/phage_genomes/A1a.fasta'
# phanotate_path = '/home/dylan33smith/.local/bin/phanotate.py'
phanotate_path = "/home/dylan33smith/src/anaconda3/envs/bioML/bin/phanotate.py"
phage_path = os.path.abspath(os.path.join(dir_path, phage_file))

In [48]:
def gene_to_protein(gene_sequence):
  """
  converts gene sequence to protein sequence
  """
  dna = gene_sequence.upper()
  dna_seq_obj = Seq(dna) # convert string to Biopython Seq object
  protein_sequence = str(dna_seq_obj.translate(to_stop=True)) # convert to protein sequence
  return protein_sequence

def phanotate_processing(phage_fasta_path, phanotate_path):
    """
    Processes a single phage genome using PHANOTATE and saves the gene predictions to a CSV.

    INPUTS:
    - input_fasta (str): Path to the single FASTA file containing the phage genome.
        - path in relation to current working directory
    - phanotate_path (str): Path to the PHANOTATE executable/script.

    OUTPUT:
    - CSV file with columns ['phage_ID', 'gene_ID', 'gene_sequence'].
    """

    # extract phage name from file (remove directory and .fasta)
    phage_name = os.path.basename(phage_fasta_path).replace('.fasta', '')
    
    # run phanotate on input fasta file
    ## shell command string to call phanotate with the input fasta file
    phanotate_shell_command = f"{phanotate_path} {phage_fasta_path}"
    ## running subprocess that executes phanotate
    process = subprocess.Popen(phanotate_shell_command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    ## standard output (gene predictions from phanotate)
    ## wats for process to finish execution and then returns the output
    stdout, _ = process.communicate()
    if process.returncode != 0:
        raise RuntimeError(f"Error running PHANOTATE: {stdout.decode()} on {phage_name}")

    # process phanotate output (skip headers)
    # stdout is a byte string
    std_splits = stdout.split(sep=b'\n')[2:]
    
    temp_tsv_path = 'temp_phanotate_output.tsv'
    with open(temp_tsv_path, 'wb') as temp_tab:
        for split in std_splits:
            temp_tab.write(split.replace(b',', b'') + b'\n')
    
    """with open(temp_tsv_path, 'w') as temp_tab:
        for split in std_splits:
            split = split.replace(b',',b'') # remove commas for pandas compatability
            temp_tab.write(split.decode('utf-8') + '\n')"""

    orfs = pd.read_csv(temp_tsv_path, sep='\t', lineterminator='\n', index_col=False)
    
    sequence = str(SeqIO.read(phage_fasta_path, 'fasta').seq)

    name_list = []
    gene_list = []
    protein_list = []
    gene_ids = []
    count = 1

    for j, strand in enumerate(orfs['FRAME']):
        start = orfs['#START'][j]
        stop = orfs['STOP'][j]
        
        if strand == '+':
            gene = sequence[start-1:stop]
        else:
            sequence_part = sequence[stop-1:start]
            gene = str(Seq(sequence_part).reverse_complement())

        protein = gene_to_protein(gene)

        name_list.append(phage_name)
        gene_list.append(gene)
        protein_list.append(protein)
        gene_ids.append(f"{phage_name}_gp{count}")
        count += 1

    # remove temp tsv file
    os.remove(temp_tsv_path)

    genebase = pd.DataFrame(list(zip(name_list, gene_ids, gene_list, protein_list)), columns=['phage_ID', 'gene_ID', 'gene_sequence', 'protein_sequence'])
    return genebase

In [50]:
# from processing import phanotate_processing

genebase = phanotate_processing(phage_path, phanotate_path)
genebase.to_csv('data/phanotate/genebase.csv', index=False)
genebase.head()

,phage_ID,gene_ID,gene_sequence,protein_sequence
0,A1a,A1a_gp1,TTAGACGCTGTGAACCTGACGTTAGAAGCCCTGGGGGAGTCTCGCG...,LDAVNLTLEALGESRVMDINTSNPSAGLARSALARNRRGLLSTGYW...
1,A1a,A1a_gp2,ATGGCGCAATCATTAGAAGGCACCATTCAGAGTCTGCTCCAGGGCG...,MAQSLEGTIQSLLQGVSQQIPRERQPGQLGAQLNMLSDPVSGLRRR...
2,A1a,A1a_gp3,ATGGCTATGTGGTGGGCTGTCGCCGCCCTGGCAGGCTCTAAGCTGC...,MAMWWAVAALAGSKLLGAGAQIEVSKARNKAVIQQTAKQLNDIALQ...
3,A1a,A1a_gp4,ATGCCTGTAATTCAACCCAACCGACAGGGTCTAAATATCGGCGGCG...,MPVIQPNRQGLNIGGVQLQANEVNLPSTVGDVAVDTSKANRLAALA...
4,A1a,A1a_gp5,ATGGCTCAGTTTCTGAACCAAGAACCGAATCCACAGGAAAAGGATT...,MAQFLNQEPNPQEKDSAKGATLKPAPESVDWNDAGDAGLNALQRSS...


In [57]:
for i, row in genebase.iterrows():
  print(row['gene_sequence'])
  print()


AttributeError: 'DataFrame' object has no attribute 'rows'

# PHANOTATE phage gene detection

In [21]:
### takes about 6 seconds for first file
def run_phanotate(phage_path, phanotate_path, output_file):
    """
    Processes a single phage genome using PHANOTATE and saves the gene predictions to a CSV.

    INPUTS:
    - input_fasta (str): Path to the single FASTA file containing the phage genome.
    - phanotate_path (str): Path to the PHANOTATE executable/script.
    - output_csv (str): Path to the output CSV file where gene predictions will be saved.

    OUTPUT:
    - CSV file with columns ['phage_ID', 'gene_ID', 'gene_sequence'].
    """

    # extract phage name from file (remove directory and .fasta)
    phage_name = os.path.basename(phage_path).replace('.fasta', '')
    
    # run phanotate on input fasta file

    raw_str = f"{phanotate_path} {phage_path}"
    process = subprocess.Popen(raw_str, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    stdout, stderr = process.communicate()

    # process phanotate output
    std_splits = stdout.split(sep=b'\n')[2:]
    return std_splits
    



In [22]:
std_splits = run_phanotate(phage_path, phanotate_path, 'test')

In [ ]:
temp_tsv_path = 'temp_phanotate_output.tsv'
with open(temp_tsv_path, 'w') as temp_tab:
    for split in std_splits:
        split = split.replace(b',',b'') # remove commas for pandas compatability
        temp_tab.write(split.decode('utf-8') + '\n')

orfs = pd.read_csv(temp_tsv_path, sep='\t', lineterminator='\n', index_col=False)
orfs.head()

,#START,STOP,FRAME,CONTIG,SCORE
0,1,552,+,A1a,-5.967169e+02
1,562,2943,+,A1a,-4.095286e+09
2,2946,3548,+,A1a,-4.242529e+02
3,3564,6248,+,A1a,-1.094128e+12
4,6298,9996,+,A1a,-1.780538e+16


In [12]:
print(orfs.iloc[0])

#START           1
STOP           552
FRAME            +
CONTIG         A1a
SCORE    -596.7169
Name: 0, dtype: object


In [6]:
sequence = str(SeqIO.read(phage_file, 'fasta').seq)

name_list = []
gene_list = []
gene_ids = []
count = 1

In [7]:
for j, strand in enumerate(orfs['FRAME']):
    start = orfs['#START'][j]
    stop = orfs['STOP'][j]
    
    # phanotate start/stop are 1-indexed
    if strand == '+':
        gene = sequence[start-1:stop]
    else:
        sequence_part = sequence[stop-1:start]
        gene = str(Seq(sequence_part).reverse_complement())

    phage_name = os.path.basename(phage_path).replace('.fasta', '')

    name_list.append(phage_name)
    gene_list.append(gene)
    gene_ids.append(f"{phage_name}_gp{count}")
    count += 1



In [24]:
for i in range(len(gene_ids)):
    if i < 1:
        print(name_list[i])
        print(gene_ids[i])
        print(gene_list[i])
        print()

print(sequence[0:20])

A1a
A1a_gp1
TTAGACGCTGTGAACCTGACGTTAGAAGCCCTGGGGGAGTCTCGCGTTATGGATATCAACACTTCAAACCCAAGCGCAGGGTTAGCACGTTCTGCACTCGCGCGTAATCGCCGAGGCCTGCTAAGCACTGGCTACTGGTTCAACGTAGTCGAGCGAGAGGTTACTCCTACGACTGACGGACTTATTAAGGTTCCGTGGAACCAGTTGGCTGTGTATGATGCGTGCTCCGACAATAAGTACGGTGTACGCAATGGGAACCTTTACGACCTGGTAGAGCAGAACGAGTACTTCGACTCACCTGTTAAAATTAAGGTAGTGCTGGACCTCAACTTTGAGGACCTGCCGGAGCACGCGGCTATGTGGATTGCAAACTACACCACTGCGCAGGTGTACCTGAACGACCTCGGCAGTGACGGCAACTACGCCAATTACGCCTCTGAGGCGGAGCGATACAAGGCCCTGGTGCTGCGCGAGCATCTGCGTAACCAGAAGTACAGCACCAGCAAGACCAGATTCGCACGTCGTATCCGTCGTGCACGCTTCATGATTTAA

TTAGACGCTGTGAACCTGAC


In [14]:
genebase = pd.DataFrame(list(zip(name_list, gene_ids, gene_list)), columns=['phage_ID', 'gene_ID', 'gene_sequence'])
genebase.to_csv('data/phanotate/genebase.csv', index=False)

In [34]:
genebase.head()

,phage_ID,gene_ID,gene_sequence
0,A1a,A1a_gp1,TTAGACGCTGTGAACCTGACGTTAGAAGCCCTGGGGGAGTCTCGCG...
1,A1a,A1a_gp2,ATGGCGCAATCATTAGAAGGCACCATTCAGAGTCTGCTCCAGGGCG...
2,A1a,A1a_gp3,ATGGCTATGTGGTGGGCTGTCGCCGCCCTGGCAGGCTCTAAGCTGC...
3,A1a,A1a_gp4,ATGCCTGTAATTCAACCCAACCGACAGGGTCTAAATATCGGCGGCG...
4,A1a,A1a_gp5,ATGGCTCAGTTTCTGAACCAAGAACCGAATCCACAGGAAAAGGATT...


# HMM RBP detection

In [58]:
dir_path = '/home/dylan33smith/projects/Yuzhen/PB_interactions/'
phage_file = 'data/phage_genomes/A1a.fasta'
hmm_path = '/home/dylan33smith/src/hmmer-3.4'
pfam_path = '/RBPdetect_phageRBPs.hmm'
xgb_path = '/RBPdetect_xgb_hmm.json'
phage_path = os.path.abspath(os.path.join(dir_path, phage_file))

In [3]:
genebase = pd.read_csv('data/phanotate/genebase_embeddings.csv')
genebase.head()


,phage_ID,gene_ID,gene_sequence,protein_sequence,protein_embedding
0,A1a,A1a_gp1,TTAGACGCTGTGAACCTGACGTTAGAAGCCCTGGGGGAGTCTCGCG...,LDAVNLTLEALGESRVMDINTSNPSAGLARSALARNRRGLLSTGYW...,"[0.04319863021373749, 0.0043669892475008965, 0..."
1,A1a,A1a_gp2,ATGGCGCAATCATTAGAAGGCACCATTCAGAGTCTGCTCCAGGGCG...,MAQSLEGTIQSLLQGVSQQIPRERQPGQLGAQLNMLSDPVSGLRRR...,"[0.0487942099571228, -0.032444849610328674, 0...."
2,A1a,A1a_gp3,ATGGCTATGTGGTGGGCTGTCGCCGCCCTGGCAGGCTCTAAGCTGC...,MAMWWAVAALAGSKLLGAGAQIEVSKARNKAVIQQTAKQLNDIALQ...,"[0.029753318056464195, -0.047349002212285995, ..."
3,A1a,A1a_gp4,ATGCCTGTAATTCAACCCAACCGACAGGGTCTAAATATCGGCGGCG...,MPVIQPNRQGLNIGGVQLQANEVNLPSTVGDVAVDTSKANRLAALA...,"[0.010439022444188595, -0.06183784827589989, -..."
4,A1a,A1a_gp5,ATGGCTCAGTTTCTGAACCAAGAACCGAATCCACAGGAAAAGGATT...,MAQFLNQEPNPQEKDSAKGATLKPAPESVDWNDAGDAGLNALQRSS...,"[0.00653857784345746, -0.06143265217542648, -0..."


In [12]:
for i in range(5):
    row = genebase.iloc[i]
    for column in genebase.columns:
        print(f"{column}: {len(str(row[column]))}")
    print()


phage_ID: 3
gene_ID: 7
gene_sequence: 552
protein_sequence: 183
protein_embedding: 22626

phage_ID: 3
gene_ID: 7
gene_sequence: 2382
protein_sequence: 793
protein_embedding: 22668

phage_ID: 3
gene_ID: 7
gene_sequence: 603
protein_sequence: 200
protein_embedding: 22561

phage_ID: 3
gene_ID: 7
gene_sequence: 2685
protein_sequence: 894
protein_embedding: 22585

phage_ID: 3
gene_ID: 7
gene_sequence: 3699
protein_sequence: 1232
protein_embedding: 22571

